# Sample Dataset for MapReduce Testing
Plain-text files students can use to test their own Map-Reduce implementations (Word Count, Inverted Index, simple frequency analysis).

**Files:** `book_01.txt`, `book_02.txt`, `book_03.txt`, `book_04.txt`, `book_05.txt`

**Exercises in this notebook**
1. Word Count: Mapper emits `(word, 1)`, Reducer sums counts per word
2. Top-K Words: use the reducer output to find the most frequent words
3. Inverted Index: Mapper emits `(word, file_name)`, Reducer builds a unique list of files per word

Click **Runtime → Run all**. The dataset is created inside the notebook, so nothing needs to be uploaded.

## Dataset files

In [ ]:
%%writefile book_01.txt
Distributed systems process large volumes of information across many machines.
Map tasks split the input and emit intermediate key value pairs.
Reduce tasks aggregate all values that share the same key.
Students can test tokenization normalization and counting accuracy.
Reliable analytics requires careful handling of noise and duplicate terms.

In [ ]:
%%writefile book_02.txt
Big data pipelines often begin with log ingestion and parsing.
A mapper can read each line and extract meaningful words.
Another mapper can emit composite keys for advanced analysis.
Reducers combine partial counts from many independent mapper outputs.
This sample file intentionally repeats words words words for testing.

In [ ]:
%%writefile book_03.txt
Cloud storage enables scalable processing for educational projects.
MapReduce supports parallel execution when data is partitioned correctly.
Intermediate results are shuffled and sorted before reduction starts.
Engineers validate correctness by comparing expected and observed totals.
Practice with small corpora helps students debug their own framework.

In [ ]:
%%writefile book_04.txt
Text analytics includes frequency analysis keyword extraction and indexing.
An inverted index maps each term to the documents that contain it.
Students should remove punctuation and normalize letter casing.
Edge cases include empty lines numbers and mixed language tokens.
Repeatable experiments improve confidence in reducer determinism.

In [ ]:
%%writefile book_05.txt
Performance tuning may involve custom partitioners and combiners.
Combiner logic can reduce network traffic before the shuffle phase.
Skewed keys can cause stragglers and unbalanced reducer workloads.
Testing with multiple files verifies cross file aggregation behavior.
This repository now includes plain text datasets for map reducer practice.

In [ ]:
%%writefile expected_word_count.txt
accuracy 1
across 1
advanced 1
aggregate 1
aggregation 1
all 1
analytics 2
an 2
and 12
another 1
are 1
balanced 1
before 1
begin 1
behavior 1
big 1
by 1
can 5
careful 1
casing 1
cause 1
cloud 1
combine 1
combiner 2
comparing 1
composite 1
confidence 1
contain 1
correctly 1
correctness 1
corpora 1
counting 1
counts 1
cross 1
custom 1
data 1
datasets 1
debug 1
determinism 1
distributed 1
documents 1
duplicate 1
each 2
edge 1
educational 1
emit 2
empty 1
enables 1
engineers 1
execution 1
expected 1
experiments 1
extract 1
extraction 1
file 2
files 2
for 3
framework 1
frequency 1
ingestion 1
includes 1
index 2
indexing 1
independent 1
information 1
input 1
intentionally 1
intermediate 2
inverted 1
involve 1
is 1
it 1
key 2
keys 2
keyword 1
language 1
large 1
letter 1
line 1
lines 1
log 1
logic 1
machines 1
many 2
map 3
mapper 3
mapreduce 1
maps 1
may 1
meaningful 1
mixed 1
multiple 1
network 1
noise 1
normalize 1
normalization 1
now 1
numbers 1
observed 1
of 3
often 1
on 1
own 1
outputs 1
parallel 1
parsing 1
partitioned 1
partitioners 1
pairs 1
performance 1
phase 1
pipelines 1
plain 1
practice 1
processing 2
projects 1
punctuation 1
read 1
reduce 2
reducer 2
reducers 2
reduction 1
reduce 1
reliable 1
remove 1
repeatable 1
repeats 1
repository 1
results 1
same 1
sample 1
scalable 1
share 1
should 1
shuffle 1
skewed 1
small 1
sorted 1
split 1
starts 1
storage 1
stragglers 1
students 3
supports 1
systems 1
tasks 2
term 1
terms 1
testing 2
text 2
that 3
the 4
this 4
to 4
tokenization 1
tokens 1
totals 1
traffic 1
tuning 1
unbalanced 1
validate 1
verifies 1
volumes 1
when 1
with 2
word 1
words 4
workloads 1

## MapReduce framework (map → shuffle → reduce)
This small framework runs any mapper and reducer. Every mapper output from every file is grouped by key in the shuffle step, so the reducer sees all values for a key across all files.

In [ ]:
import re, glob, os
from collections import defaultdict

files = sorted(glob.glob("book_*.txt"))

def tokenize(line):
    # lowercase and remove punctuation
    return re.findall(r"[a-z0-9]+", line.lower())

def map_phase(files, mapper):
    for path in files:
        for line in open(path):
            yield from mapper(os.path.basename(path), line)

def shuffle(pairs):
    groups = defaultdict(list)
    for key, value in pairs:
        groups[key].append(value)
    return groups

def reduce_phase(groups, reducer):
    return {key: reducer(key, values) for key, values in sorted(groups.items())}

def run_mapreduce(files, mapper, reducer):
    return reduce_phase(shuffle(map_phase(files, mapper)), reducer)

## 1. Word Count
- Mapper emits `(word, 1)`
- Reducer sums counts per word

In [ ]:
def wc_mapper(file_name, line):
    for word in tokenize(line):
        yield (word, 1)

def wc_reducer(word, counts):
    return sum(counts)

word_counts = run_mapreduce(files, wc_mapper, wc_reducer)

print("Unique words:", len(word_counts))
print("Total words :", sum(word_counts.values()))
print()
for word, count in word_counts.items():
    print(word, count)

## 2. Top-K Words
Use the reducer output (`word_counts`) to find the most frequent words.

In [ ]:
K = 10   # change K to see more or fewer words

top_k = sorted(word_counts.items(), key=lambda item: (-item[1], item[0]))[:K]

print(f"Top {K} words")
for rank, (word, count) in enumerate(top_k, start=1):
    print(f"{rank:>2}. {word:<12} {count}")

## 3. Inverted Index
- Mapper emits `(word, file_name)`
- Reducer builds a unique list of files per word

In [ ]:
def ii_mapper(file_name, line):
    for word in tokenize(line):
        yield (word, file_name)

def ii_reducer(word, file_names):
    return sorted(set(file_names))     # unique list of files

inverted_index = run_mapreduce(files, ii_mapper, ii_reducer)

for word, file_list in inverted_index.items():
    print(f"{word:<15} {file_list}")

## Validation: key aggregation across multiple mapper outputs
The files overlap, so some words must appear in more than one file. These checks confirm the reducers combined values from different files correctly.

In [ ]:
# Words that appear in several files
multi = {w: f for w, f in inverted_index.items() if len(f) > 1}
print(f"{len(multi)} words appear in more than one file. Examples:")
for w in ["and", "the", "can", "students", "with", "reducer"]:
    print(f"  {w:<10} count={word_counts[w]:<3} files={inverted_index[w]}")

# Check 1: total count equals the number of words in all files
raw_total = sum(len(tokenize(line)) for p in files for line in open(p))
assert raw_total == sum(word_counts.values())

# Check 2: each word's count equals its occurrences counted file by file
for w in word_counts:
    per_file = sum(tokenize(line).count(w) for p in files for line in open(p))
    assert per_file == word_counts[w], w

# Check 3: every word in the index points to files that really contain it
for w, fl in inverted_index.items():
    for name in fl:
        assert any(w in tokenize(line) for line in open(name)), (w, name)

print("\nAll checks passed.")

## Save and download the results

In [ ]:
with open("word_count_output.txt", "w") as f:
    for w, n in word_counts.items():
        f.write(f"{w} {n}\n")

with open("top_k_output.txt", "w") as f:
    for w, n in top_k:
        f.write(f"{w} {n}\n")

with open("inverted_index_output.txt", "w") as f:
    for w, fl in inverted_index.items():
        f.write(f"{w}\t{', '.join(fl)}\n")

try:
    from google.colab import files as colab_files
    for name in ["word_count_output.txt", "top_k_output.txt", "inverted_index_output.txt"]:
        colab_files.download(name)
except ImportError:
    print("Saved the three output files.")

## Optional: compare with `expected_word_count.txt`
Note: the provided expected file does not match the actual text of the five books (it lists words that never appear, such as `balanced`, and different counts for others), so differences are expected here. The checks above verify the results against the real files.

In [ ]:
expected = {}
for line in open("expected_word_count.txt"):
    parts = line.split()
    if len(parts) == 2:
        expected[parts[0]] = max(expected.get(parts[0], 0), int(parts[1]))

diffs = [(w, expected.get(w), word_counts.get(w))
         for w in sorted(set(expected) | set(word_counts))
         if expected.get(w) != word_counts.get(w)]

print(f"{len(diffs)} words differ from the expected file")
print(f"{'word':<12} {'expected':>8} {'actual':>8}")
for w, e, a in diffs:
    print(f"{w:<12} {str(e):>8} {str(a):>8}")